# CattleSense AI — Research v1.3: Rumination & Uncertainty

**Aim:** Evaluate whether **training-only class rebalancing** improves four-class cattle activity recognition on unseen cows, without looking at test-cow results during model/threshold selection.

Three comparable candidates:
1. **Random Forest, 17 features** — same baseline as v1.2.
2. **Extra Trees, 27 features** — same candidate as v1.2.
3. **Rebalanced Random Forest, 17 features** — oversamples minority-class *training windows only*, capped at the median class support.

For each of six held-out test cows: use one different cow for validation, and the remaining four cows for training. Model and rejection threshold are picked using **validation data only**. Results are descriptive with only six animals, not proof of livestock welfare outcomes. Public data are from Japanese Black beef cattle, not dairy cows.

The notebook exports metrics and figures, **not a deployed model**. Keep your original working website unchanged.

## Step 1. Prepare Python libraries

In [ ]:
import re, zipfile, urllib.request, json, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, confusion_matrix, recall_score
from IPython.display import display

SEED=42
np.random.seed(SEED)
OUT=Path('cattlesense_v13_results'); OUT.mkdir(exist_ok=True)
KEEP=['RES','RUS','MOV','GRZ']
print('Ready. Results will be saved to', OUT.resolve())
from sklearn.utils import resample
from collections import Counter
from scipy.optimize import minimize_scalar

## Step 2. Download dataset ZIP (official Zenodo record)
If the download fails, manually upload `JapaneseBlackBeefData.zip` to the Colab Files panel, then rerun this one cell.

In [ ]:
DATA_URL='https://zenodo.org/records/5849025/files/JapaneseBlackBeefData.zip?download=1'
zip_path=Path('JapaneseBlackBeefData.zip')
if not zip_path.exists():
    print('Downloading official ZIP from Zenodo...')
    try:
        urllib.request.urlretrieve(DATA_URL, zip_path)
    except Exception as exc:
        raise RuntimeError('Download failed. Upload JapaneseBlackBeefData.zip manually using the Colab Files sidebar, then retry.') from exc
if not zipfile.is_zipfile(zip_path):
    raise ValueError('The downloaded file is not a valid ZIP. Delete it and download the dataset from Zenodo.')
with zipfile.ZipFile(zip_path) as z:
    csv_names=[name for name in z.namelist() if re.search(r'(?:^|/)cow[1-6]\.csv$',name,re.I)]
assert len(csv_names)==6, f'Expected six cow CSVs; found {csv_names}'
print('ZIP size (MB):', round(zip_path.stat().st_size/1e6,2))
print('Data CSVs:',csv_names)

## Step 3. Load the six cows and automatically fix milliseconds to seconds

In [ ]:
def normalized(s): return re.sub(r'[^a-z0-9]','',str(s).lower())

def load_cow(file_handle, cow_id):
    df=pd.read_csv(file_handle,low_memory=False)
    norms={normalized(c):c for c in df.columns}
    needed={'x':'accx','y':'accy','z':'accz','label':'label','time':'timestampunix'}
    mapping={new:next((orig for k,orig in norms.items() if k.startswith(prefix)),None) for new,prefix in needed.items()}
    if any(v is None for v in mapping.values()):
        raise ValueError(f'Unrecognized columns in cow {cow_id}: {list(df.columns)}')
    df=df.rename(columns={v:k for k,v in mapping.items()})[['time','x','y','z','label']].copy()
    for col in ['time','x','y','z']:
        df[col]=pd.to_numeric(df[col],errors='coerce')
    # Dataset UNIX timestamps are milliseconds. Subtract origin first to avoid float precision loss.
    df['time']=(df['time'].astype('float64')-float(df['time'].iloc[0]))/1000.0
    df['label']=df['label'].astype(str).str.strip().str.upper()
    df['cow']=cow_id
    return df

all_cows={}
with zipfile.ZipFile(zip_path) as z:
    for name in csv_names:
        cow=int(re.search(r'cow([1-6])\.csv$',name,re.I).group(1))
        with z.open(name) as f: all_cows[cow]=load_cow(f,cow)
print('Raw samples / cow:',{c:len(d) for c,d in all_cows.items()})
print('All labels:',sorted(set().union(*(set(d.label.unique()) for d in all_cows.values()))))
for cow,df in sorted(all_cows.items()):
    diffs=df.time.diff().dropna()
    positive=diffs[diffs>0]
    print(f'Cow {cow}: positive timestamp delta (median) = {positive.median() if len(positive) else "unavailable"}')

## Step 4. Inspect activity imbalance for each cow

In [ ]:
dist=pd.DataFrame({f'cow_{k}':v.label.value_counts() for k,v in all_cows.items()}).fillna(0).astype(int)
display(dist.reindex(KEEP,fill_value=0))
ax=dist.reindex(KEEP,fill_value=0).T.plot.bar(figsize=(9,4),title='Number of labeled raw samples by cow')
ax.set_ylabel('Samples'); plt.tight_layout(); plt.savefig(OUT/'class_distribution.png',dpi=160); plt.show()

## Step 5. Create valid 2-second windows and both feature sets
The time-gap and label checks avoid windows with missing discontinuities or mixed behaviour labels.

In [ ]:
FEATURES_BASE = ['acc_mean_x','acc_mean_y','acc_mean_z','acc_std_x','acc_std_y','acc_std_z',
 'acc_min_x','acc_min_y','acc_min_z','acc_max_x','acc_max_y','acc_max_z',
 'acc_energy_x','acc_energy_y','acc_energy_z','magnitude_mean','magnitude_std']
FEATURES_EXTRA = ['median_x','median_y','median_z','mean_abs_diff_x','mean_abs_diff_y','mean_abs_diff_z',
                  'magnitude_median','magnitude_range','magnitude_mean_abs_diff','magnitude_energy']

def base_features(a):
    a=np.asarray(a,dtype=np.float32)
    mag=np.linalg.norm(a,axis=1)
    return np.r_[a.mean(axis=0), a.std(axis=0), a.min(axis=0), a.max(axis=0),
                 (a*a).mean(axis=0), mag.mean(),mag.std()].astype(np.float32)

def extended_features(a):
    a=np.asarray(a,dtype=np.float32)
    mag=np.linalg.norm(a,axis=1)
    d=np.diff(a,axis=0)
    dm=np.diff(mag)
    extra=np.r_[np.median(a,axis=0),np.abs(d).mean(axis=0),
                 np.median(mag),np.ptp(mag),np.abs(dm).mean(),np.mean(mag**2)]
    return np.r_[base_features(a),extra].astype(np.float32)

def make_windows(df,width=50,stride=25,max_gap_seconds=0.12):
    a=df[['x','y','z']].to_numpy(dtype=np.float32)
    times=df.time.to_numpy(dtype=np.float64)
    labels=df.label.to_numpy()
    xb,xe,ys=[],[],[]
    rejected={'label':0,'missing':0,'time_gap':0}
    for start in range(0,len(df)-width+1,stride):
        target=labels[start:start+width]
        data=a[start:start+width]
        t=times[start:start+width]
        if target[0] not in KEEP or not np.all(target==target[0]):
            rejected['label']+=1; continue
        if not (np.isfinite(data).all() and np.isfinite(t).all()):
            rejected['missing']+=1; continue
        dt=np.diff(t)
        if np.any(dt<=0) or np.any(dt>max_gap_seconds):
            rejected['time_gap']+=1; continue
        xb.append(base_features(data));xe.append(extended_features(data));ys.append(target[0])
    return (np.asarray(xb,dtype=np.float32).reshape(-1,17),
            np.asarray(xe,dtype=np.float32).reshape(-1,27),np.asarray(ys),rejected)

windows={}
for cow,df in sorted(all_cows.items()):
    Xb,Xe,y,rej=make_windows(df)
    windows[cow]={'base':Xb,'extended':Xe,'y':y}
    print(f'Cow {cow}: {len(y)} windows | classes={dict(pd.Series(y).value_counts())} | excluded={rej}')
    if len(y)==0: raise ValueError(f'No usable windows for cow {cow}.')
print('Feature lengths:',len(FEATURES_BASE),len(FEATURES_BASE+FEATURES_EXTRA))

## Step 6. Compare models fairly on six held-out cows

**Keep these rules:** Never oversample validation/test windows. Compare models on identical train/validation/test cows. The random forest and Extra Trees configurations here match v1.2. Oversampling is a new *research candidate*, not automatically an improvement.

The rebalancing strategy uses only training windows and upsamples minority classes to a cap equal to the median class count, without downsampling the majority class. We log original and resulting class distributions.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, log_loss

LABELS=KEEP
CANDIDATES=[('Random Forest (17)','base'),('Extra Trees (27)','extended'),('RF + train oversampling (17)','base')]
CUTOFFS=[0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
MIN_VALIDATION_COVERAGE=0.40


def stack(cows,field):
    return (np.concatenate([windows[i][field] for i in cows]),
            np.concatenate([windows[i]['y'] for i in cows]))


def oversample_training(X,y,seed):
    rng=np.random.default_rng(seed)
    counts=Counter(y)
    cap=int(np.median(list(counts.values())))
    indices=np.arange(len(y)).tolist()
    for label,n in sorted(counts.items()):
        if n<cap:
            pool=np.flatnonzero(y==label)
            indices.extend(rng.choice(pool, size=cap-n, replace=True).tolist())
    indices=np.asarray(indices)
    rng.shuffle(indices)
    return X[indices],y[indices],dict(counts),dict(Counter(y[indices]))


def make_model(name):
    if name in ['Random Forest (17)','RF + train oversampling (17)']:
        return RandomForestClassifier(n_estimators=150,max_depth=18,
            class_weight='balanced_subsample',random_state=SEED,n_jobs=-1)
    if name=='Extra Trees (27)':
        return ExtraTreesClassifier(n_estimators=200,min_samples_leaf=2,
            class_weight='balanced',random_state=SEED,n_jobs=-1)
    raise KeyError(name)


def p_aligned(model,X,classes=LABELS):
    raw=model.predict_proba(X)
    result=np.zeros((len(X),len(classes)),dtype=float)
    for j,lab in enumerate(classes):
        if lab in model.classes_:
            result[:,j]=raw[:,list(model.classes_).index(lab)]
    return result


def temperature_prob(probs,T):
    # Temperature scaling of probabilities; uses validation data ONLY for fitting T.
    logits=np.log(np.clip(probs,1e-12,1)) / float(T)
    logits=logits-logits.max(axis=1,keepdims=True)
    ex=np.exp(logits)
    return ex/ex.sum(axis=1,keepdims=True)


def fit_temperature(p_val,y_val):
    targets=np.array([LABELS.index(x) for x in y_val])
    def objective(log_t):
        p=temperature_prob(p_val,np.exp(log_t))
        return -np.log(np.clip(p[np.arange(len(targets)),targets],1e-12,1)).mean()
    opt=minimize_scalar(objective,bounds=(-2.3,2.3),method='bounded')
    return float(np.exp(opt.x))


def choose_threshold(p_val,y_val):
    pred=np.array(LABELS)[p_val.argmax(axis=1)]
    conf=p_val.max(axis=1)
    feasible=[]
    for cutoff in CUTOFFS:
        accepted=conf>=cutoff
        coverage=float(accepted.mean())
        acc=float((pred[accepted]==y_val[accepted]).mean()) if accepted.any() else None
        if accepted.any() and coverage>=MIN_VALIDATION_COVERAGE:
            feasible.append((acc,coverage,-cutoff,cutoff))
    if not feasible:
        return min(CUTOFFS)
    return sorted(feasible,reverse=True)[0][3]


def selective_stats(p,y,cutoff):
    pred=np.array(LABELS)[p.argmax(axis=1)]
    taken=p.max(axis=1)>=cutoff
    return {'coverage':float(taken.mean()),'accepted_count':int(taken.sum()),
            'accepted_accuracy':float((pred[taken]==y[taken]).mean()) if taken.any() else None}

rows=[]; recalls=[]; confidences=[]; val_choices=[]; supports=[]
for test_cow in sorted(windows):
    others=[i for i in sorted(windows) if i!=test_cow]
    val_cow=others[0]
    training=[i for i in others if i!=val_cow]
    candidates=[]
    print(f'\nTEST cow {test_cow} | VALIDATION cow {val_cow} | TRAIN {training}')
    for name,field in CANDIDATES:
        Xtr,ytr=stack(training,field)
        Xv,yv=stack([val_cow],field)
        Xt,yt=stack([test_cow],field)
        if name=='RF + train oversampling (17)':
            Xtr,ytr,before,after=oversample_training(Xtr,ytr,seed=SEED+test_cow)
        else:
            before=dict(Counter(ytr));after=before
        supports.append({'test_cow':test_cow,'model':name,'train_before':str(before),
                         'train_after':str(after)})
        model=make_model(name)
        model.fit(Xtr,ytr)
        pv=p_aligned(model,Xv)
        pt=p_aligned(model,Xt)
        val_pred=np.array(LABELS)[pv.argmax(axis=1)]
        val_f1=float(f1_score(yv,val_pred,labels=LABELS,average='macro',zero_division=0))
        temperature=fit_temperature(pv,yv)
        pv_cal=temperature_prob(pv,temperature)
        cutoff=choose_threshold(pv_cal,yv)
        pt_cal=temperature_prob(pt,temperature)
        y_pred=np.array(LABELS)[pt.argmax(axis=1)]
        test_acc=float(accuracy_score(yt,y_pred))
        test_f1=float(f1_score(yt,y_pred,labels=LABELS,average='macro',zero_division=0))
        rus_mask=yt=='RUS'
        rus_recall=float((y_pred[rus_mask]=='RUS').mean()) if rus_mask.any() else None
        stats=selective_stats(pt_cal,yt,cutoff)
        rows.append({'test_cow':test_cow,'validation_cow':val_cow,'train_cows':str(training),
                     'model':name,'test_windows':len(yt),'val_macro_f1':val_f1,
                     'test_accuracy':test_acc,'test_macro_f1':test_f1,
                     'rus_support':int(rus_mask.sum()),'rus_recall':rus_recall,
                     'temperature_fit_on_val':temperature,'cutoff_chosen_on_val':cutoff,**stats})
        for label in LABELS:
            mask=yt==label
            recalls.append({'test_cow':test_cow,'model':name,'class':label,
                            'actual_support':int(mask.sum()),
                            'recall':float((y_pred[mask]==label).mean()) if mask.any() else None})
        for fixed_cutoff in CUTOFFS:
            confidences.append({'test_cow':test_cow,'model':name,'threshold':fixed_cutoff,
                                'temperature_fit_on_val':temperature,
                                **selective_stats(pt_cal,yt,fixed_cutoff)})
        candidates.append((val_f1,name))
        print(f'  {name}: test accuracy {test_acc:.2%}; macro F1 {test_f1:.2%}; '
              f'RUS recall {rus_recall if rus_recall is not None else "N/A"}')
    selected=sorted(candidates,key=lambda t:(-t[0],t[1]))[0][1]
    val_choices.append({'test_cow':test_cow,'selected_model_using_validation_only':selected})

comparison=pd.DataFrame(rows)
recall_df=pd.DataFrame(recalls)
confidence_df=pd.DataFrame(confidences)
selection=pd.DataFrame(val_choices).merge(comparison,left_on=['test_cow','selected_model_using_validation_only'],right_on=['test_cow','model'])
train_support=pd.DataFrame(supports)
for name,df in [('model_comparison_by_cow',comparison),('class_recall_by_cow',recall_df),
                ('confidence_threshold_diagnostics',confidence_df),('validation_selected_per_cow',selection),
                ('train_resampling_support',train_support)]:
    df.to_csv(OUT/f'{name}.csv',index=False)
print('\nDetailed results:')
display(comparison.round(4))

## Step 7. Research comparison charts

Use **macro F1** as the primary metric since the class distribution is imbalanced. Missing rumination labels for a cow mean rumination recall is **N/A**, not 0%. Do **not** compare means as evidence of statistical significance with only six cows.

In [ ]:
means=comparison.groupby('model').agg(
    mean_accuracy=('test_accuracy','mean'),
    mean_macro_f1=('test_macro_f1','mean'),
    mean_rus_recall_supported=('rus_recall','mean'),
    mean_coverage_selected=('coverage','mean'),
    mean_acc_accepted=('accepted_accuracy','mean')).reset_index()
display(means.round(4))
means.to_csv(OUT/'model_summary.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(13,4.3))
for name,g in comparison.groupby('model'):
    g=g.sort_values('test_cow')
    axes[0].plot(g.test_cow,g.test_macro_f1,marker='o',label=name)
    axes[1].plot(g.test_cow,g.test_accuracy,marker='o',label=name)
for ax,title in zip(axes,['Macro F1 on held-out cows','Accuracy on held-out cows']):
    ax.set(title=title,xlabel='Held-out cow',ylim=(0,1));ax.grid(alpha=.25)
axes[0].set_ylabel('Score');axes[0].legend(fontsize=8)
plt.tight_layout();plt.savefig(OUT/'model_comparison.png',dpi=160);plt.show()

rus=recall_df[(recall_df['class']=='RUS') & (recall_df.actual_support>0)]
fig,ax=plt.subplots(figsize=(10,4))
for name,g in rus.groupby('model'):
    ax.plot(g.test_cow,g.recall,marker='o',label=name)
ax.set(title='Rumination recall (only cows with RUS examples)',xlabel='Held-out cow',ylabel='RUS recall',ylim=(0,1))
ax.legend(fontsize=8);ax.grid(alpha=.25)
plt.tight_layout();plt.savefig(OUT/'rumination_recall.png',dpi=160);plt.show()

fig,ax=plt.subplots(figsize=(9,4))
for name,g in confidence_df.groupby('model'):
    series=g.groupby('threshold').coverage.mean()
    ax.plot(series.index,series.values,marker='o',label=name)
ax.set(title='Coverage after validation-fit temperature scaling',xlabel='Rejection threshold',ylabel='Average coverage',ylim=(0,1))
ax.legend(fontsize=8);ax.grid(alpha=.25)
plt.tight_layout();plt.savefig(OUT/'coverage_by_threshold.png',dpi=160);plt.show()

print('Chosen models per test cow (based only on validation macro F1):')
display(selection[['test_cow','selected_model_using_validation_only','val_macro_f1','test_macro_f1','rus_recall','coverage','accepted_accuracy']].round(4))

## Step 8. Export the experiment results ZIP

This ZIP contains **experiment evidence**, not a deployment-ready classifier. We can select the final model only after reviewing v1.3 results, verifying the old baseline and making a dedicated deployment/training split. Raw data are intentionally excluded.

In [ ]:
from zipfile import ZipFile
summary={
  'experiment':'CattleSense v1.3: training-only rebalancing and validation-selected rejection',
  'dataset':'Japanese Black Beef Cow Behavior Classification Dataset v2',
  'dataset_url':'https://zenodo.org/records/5849025',
  'activity_labels':LABELS,
  'n_cows':len(windows),
  'window_size':50,'stride':25,'sample_rate_hz':25,
  'features_base':FEATURES_BASE,'features_extended':FEATURES_BASE+FEATURES_EXTRA,
  'candidates':[c[0] for c in CANDIDATES],
  'selection':'validation macro F1, one held-out validation cow per fold',
  'rejection_policy':'validation-fitted temperature scaling and a validation-selected cutoff with at least 40% validation coverage',
  'means':means.to_dict(orient='records'),
  'caveats':['Only six beef cattle; not validated on dairy cows',
             'Overlapping windows reduce effective independence',
             'Cross-animal results are exploratory; no separate external validation set',
             'Selectivity on individual cows may be poorly calibrated',
             'Test folds already used in earlier exploration; do not claim pristine independent test set',
             'No on-device memory, latency or power measurements'],
  'created_utc':datetime.now(timezone.utc).isoformat()
}
(OUT/'research_summary_v13.json').write_text(json.dumps(summary,indent=2,default=str),encoding='utf-8')
(OUT/'LIMITATIONS.txt').write_text('\n'.join(summary['caveats']),encoding='utf-8')
archive=Path('CattleSense_v1_3_Research_Results.zip')
with ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print('Saved:',archive.resolve(),'size:',round(archive.stat().st_size/1024,1),'KB')
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Local environment: download ZIP manually from the current folder.')